# Embedding & Reranker 연결 테스트
- Embedding: 192.168.1.166:58001
- Reranker: 192.168.1.166:58002

In [1]:
import requests

EMBEDDING_URL = "http://192.168.1.166:58001"
RERANKER_URL  = "http://192.168.1.166:58002"

## 1. Health Check

In [2]:
for name, url in [("embedding", EMBEDDING_URL), ("reranker", RERANKER_URL)]:
    try:
        r = requests.get(f"{url}/health", timeout=5)
        print(f"[{name}] status: {r.status_code}")
    except Exception as e:
        print(f"[{name}] 연결 실패: {e}")

[embedding] status: 200
[reranker] status: 200


## 2. Embedding 테스트

In [3]:
texts = [
    "입찰 참가 자격은 중소기업으로 한정한다.",
    "계약 금액은 부가세 포함 금액으로 한다.",
    "납품 기한은 계약일로부터 30일 이내로 한다."
]

response = requests.post(
    f"{EMBEDDING_URL}/v1/embeddings",
    json={
        "model": "/model",
        "input": texts
    }
)

result = response.json()
embeddings = [item["embedding"] for item in result["data"]]

print(f"임베딩 개수: {len(embeddings)}")
print(f"벡터 차원: {len(embeddings[0])}")
print(f"첫 번째 벡터 앞 5개: {embeddings[0][:5]}")

임베딩 개수: 3
벡터 차원: 1024
첫 번째 벡터 앞 5개: [-0.085205078125, 0.050445556640625, -0.01355743408203125, -0.0060882568359375, -0.002895355224609375]


## 3. Reranker 테스트

In [4]:
query = "계약 기간은 얼마나 되나요?"

passages = [
    "입찰 참가 자격은 중소기업으로 한정한다.",
    "계약 금액은 부가세 포함 금액으로 한다.",
    "납품 기한은 계약일로부터 30일 이내로 한다."
]

response = requests.post(
    f"{RERANKER_URL}/v1/score",
    json={
        "model": "/model",
        "text_1": query,
        "text_2": passages
    }
)

result = response.json()
scores = [item["score"] for item in result["data"]]

print(f"Query: {query}\n")
for passage, score in sorted(zip(passages, scores), key=lambda x: x[1], reverse=True):
    print(f"score: {score:.4f} | {passage}")

Query: 계약 기간은 얼마나 되나요?

score: 0.0752 | 납품 기한은 계약일로부터 30일 이내로 한다.
score: 0.0002 | 계약 금액은 부가세 포함 금액으로 한다.
score: 0.0000 | 입찰 참가 자격은 중소기업으로 한정한다.


## 4. 코사인 유사도 확인 (임베딩 품질 검증)

In [5]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

query_response = requests.post(
    f"{EMBEDDING_URL}/v1/embeddings",
    json={"model": "/model", "input": [query]}
)
query_emb = query_response.json()["data"][0]["embedding"]

print(f"Query: {query}\n")
for passage, emb in sorted(
    zip(passages, embeddings),
    key=lambda x: cosine_similarity(query_emb, x[1]),
    reverse=True
):
    sim = cosine_similarity(query_emb, emb)
    print(f"similarity: {sim:.4f} | {passage}")

Query: 계약 기간은 얼마나 되나요?

similarity: 0.6490 | 납품 기한은 계약일로부터 30일 이내로 한다.
similarity: 0.5569 | 계약 금액은 부가세 포함 금액으로 한다.
similarity: 0.4298 | 입찰 참가 자격은 중소기업으로 한정한다.
